# Generation-fenced token dropping

This bounded end-to-end example composes `GenerationFencedDropLifecycle` with a real `LMCacheRequestStream.modify_kv()` operation. It runs one prefill, retrieves the cached KV through LMCache, removes the middle half of its chunks, stores the compacted KV, and resumes decode.

The lifecycle does not choose tokens or move KV by itself. The serving adapter owns those effects and advances the lifecycle only when each effect has actually completed. The final evidence must show one applied effect, a consumed plan, an idempotent duplicate completion, and a stale late completion after request-generation reuse.

## Start the services

Start LMCache first:

```bash
lmcache server \
  --l1-size-gb 8 \
  --eviction-policy LRU \
  --chunk-size 256 \
  --port 6555 \
  --http-port 8080 \
  --shm-name lmcache_generation_fenced_drop_e2e \
  --no-l1-use-lazy \
  --supported-transfer-mode auto
curl -sf http://localhost:8080/healthcheck
```

Then start vLLM on one GPU:

```bash
CUDA_VISIBLE_DEVICES=0 VLLM_ENABLE_V1_MULTIPROCESSING=0 \
vllm serve Qwen/Qwen3-0.6B \
  --port 8000 \
  --served-model-name Qwen/Qwen3-0.6B \
  --no-enable-prefix-caching \
  --enforce-eager \
  --max-model-len 2048 \
  --gpu-memory-utilization 0.5 \
  --kv-transfer-config '{"kv_connector":"LMCacheMPConnector","kv_role":"kv_both","kv_connector_extra_config":{"lmcache.mp.port":6555}}' \
  --trust-remote-code \
  --return-tokens-as-token-ids
curl -sf http://localhost:8000/v1/models
```

When an offline deployment resolves the model ID to a local snapshot path, set `HF_MODEL_NAME` and `LMCACHE_MODEL_NAME` to that path before executing the notebook. `LMCACHE_MODEL_NAME` must match the connector's logged `cache_model_name`; `VLLM_SERVED_MODEL_NAME` remains the public API alias.

In [ ]:
# SPDX-License-Identifier: Apache-2.0
"""Run one real cache edit under generation-fenced lifecycle control."""

# Standard
from dataclasses import asdict, dataclass
from pathlib import Path
import json
import os
import sys
import time

# Third Party
import torch
from transformers import AutoConfig, AutoTokenizer

# First Party
import lmcache.sdk as lmc_sdk
from lmcache.sdk.drop_lifecycle import (
    DropCompletionDisposition,
    DropOperationCompletion,
    DropOperationKey,
    DropOperationState,
    GenerationFencedDropLifecycle,
)

example_candidates = (Path("examples/token_dropping"), Path.cwd())
example_dir = next(
    (path for path in example_candidates if (path / "utils.py").is_file()),
    None,
)
if example_dir is None:
    raise FileNotFoundError("run from the repository root or examples/token_dropping")
sys.path.insert(0, str(example_dir.resolve()))
from utils import make_post_completion, rerotate_k_cache  # noqa: E402

In [ ]:
served_model_name = os.environ.get("VLLM_SERVED_MODEL_NAME", "Qwen/Qwen3-0.6B")
hf_model_name = os.environ.get("HF_MODEL_NAME", served_model_name)
cache_model_name = os.environ.get("LMCACHE_MODEL_NAME", served_model_name)
vllm_url = "http://localhost:8000"
lmcache_url = "http://localhost:8080"
lmcache_mq_url = "tcp://localhost:6555"
chunk_size = 256
prompt_tokens = chunk_size * 4
max_tokens = 32
timeout = 60.0
policy_revision = "middle-chunk-drop-v1"
work_device = torch.device("cpu")

In [ ]:
@dataclass
class PendingFencedEdit:
    lifecycle: GenerationFencedDropLifecycle | None = None
    key: DropOperationKey | None = None
    completion: DropOperationCompletion | None = None
    source_tokens: int = 0
    kept_tokens: int = 0


def require_disposition(
    actual: DropCompletionDisposition,
    expected: DropCompletionDisposition,
    phase: str,
) -> None:
    if actual != expected:
        raise RuntimeError(f"{phase}: expected {expected.value}, got {actual.value}")


def run_fenced_drop(
    request_stream: lmc_sdk.request.LMCacheRequestStream,
    model_config: AutoConfig,
) -> PendingFencedEdit:
    """Retrieve, compact, and store one request's KV under a fence."""
    pending = PendingFencedEdit()

    def edit(
        tensors: dict[lmc_sdk.context.LMCacheSDKCacheKind, torch.Tensor],
        token_source: list[int],
    ) -> tuple[torch.Tensor, list[int]]:
        kv_tensor = tensors[lmc_sdk.context.LMCacheSDKCacheKind.KV]
        cached_tokens = int(kv_tensor.shape[2])
        num_chunks = (cached_tokens + chunk_size - 1) // chunk_size
        if num_chunks < 3:
            raise ValueError("at least three cached chunks are required")

        topology_fingerprint = (
            f"shape={tuple(kv_tensor.shape)};dtype={kv_tensor.dtype};"
            f"chunk_size={chunk_size}"
        )
        lifecycle = GenerationFencedDropLifecycle(
            request_id=request_stream.request_stream_id,
            request_generation=0,
            topology_fingerprint=topology_fingerprint,
            policy_revision=policy_revision,
        )
        key = lifecycle.begin_operation(
            drop_round=0,
            source_kv_revision=0,
            accepted_seq_len=cached_tokens,
        )
        require_disposition(
            lifecycle.advance(key, DropOperationState.PLAN_COMPUTING),
            DropCompletionDisposition.RECORDED,
            "plan computing",
        )

        drop_count = num_chunks // 2
        drop_start = max(1, (num_chunks - drop_count) // 2)
        drop_start = min(drop_start, num_chunks - 1 - drop_count)
        lo = drop_start * chunk_size
        hi = min((drop_start + drop_count) * chunk_size, cached_tokens)
        keep_idx = torch.cat([torch.arange(lo), torch.arange(hi, cached_tokens)])
        kept_ids = list(token_source[:lo]) + list(token_source[hi:cached_tokens])
        compacted = rerotate_k_cache(
            kv_tensor[:, :, keep_idx, :].clone().to(work_device),
            old_positions=keep_idx.to(work_device),
            new_positions=torch.arange(
                keep_idx.numel(), device=work_device, dtype=torch.long
            ),
            model_config=model_config,
        ).cpu()

        require_disposition(
            lifecycle.advance(key, DropOperationState.PLAN_READY),
            DropCompletionDisposition.RECORDED,
            "plan ready",
        )
        require_disposition(
            lifecycle.advance(key, DropOperationState.PLAN_VALIDATED),
            DropCompletionDisposition.RECORDED,
            "plan validated",
        )
        require_disposition(
            lifecycle.advance(key, DropOperationState.DEACTIVATION_SUBMITTED),
            DropCompletionDisposition.RECORDED,
            "cache edit submitted",
        )

        completion = DropOperationCompletion(
            key=key,
            topology_fingerprint=topology_fingerprint,
            policy_revision=policy_revision,
            accepted_seq_len=cached_tokens,
        )
        pending.lifecycle = lifecycle
        pending.key = key
        pending.completion = completion
        pending.source_tokens = cached_tokens
        pending.kept_tokens = len(kept_ids)
        return compacted, kept_ids

    request_stream.modify_kv(edit, timeout=timeout)
    if pending.lifecycle is None or pending.key is None or pending.completion is None:
        raise RuntimeError("cache edit returned without lifecycle metadata")

    require_disposition(
        pending.lifecycle.record_remote_completion(pending.completion),
        DropCompletionDisposition.RECORDED,
        "remote completion",
    )
    require_disposition(
        pending.lifecycle.mark_local_visible(pending.completion),
        DropCompletionDisposition.APPLIED,
        "local visibility",
    )
    require_disposition(
        pending.lifecycle.consume(pending.key),
        DropCompletionDisposition.RECORDED,
        "plan consumption",
    )
    return pending

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(hf_model_name, trust_remote_code=True)
model_config = AutoConfig.from_pretrained(hf_model_name, trust_remote_code=True)
seed = tokenizer.encode(
    "Generation fences keep delayed cache edits from crossing request lifetimes. ",
    add_special_tokens=False,
)
prompt = (seed * ((prompt_tokens + len(seed) - 1) // len(seed)))[:prompt_tokens]
post_completion = make_post_completion(vllm_url, served_model_name, timeout)
ctx = lmc_sdk.kvcache.connect(
    url=lmcache_mq_url,
    http_url=lmcache_url,
    model_name=cache_model_name,
    timeout=timeout,
)

try:
    request_stream = lmc_sdk.request.create_request(
        contexts=[ctx],
        post_completion=post_completion,
        prompt_token_ids=prompt,
        cache_salt=f"generation-fenced-drop-{time.time_ns()}",
    )
    prefill = request_stream.generate(
        {"max_tokens": 1, "temperature": 0.0, "ignore_eos": True}
    )
    fenced = run_fenced_drop(request_stream, model_config)
    assert fenced.lifecycle is not None
    assert fenced.key is not None
    assert fenced.completion is not None

    duplicate = fenced.lifecycle.record_remote_completion(fenced.completion)
    require_disposition(
        duplicate,
        DropCompletionDisposition.DUPLICATE_NOOP,
        "duplicate completion",
    )
    fenced.lifecycle.advance_generation(
        1,
        topology_fingerprint=fenced.completion.topology_fingerprint,
        policy_revision=policy_revision,
    )
    late = fenced.lifecycle.record_remote_completion(fenced.completion)
    require_disposition(
        late,
        DropCompletionDisposition.STALE_NOOP,
        "late old-generation completion",
    )

    decode = request_stream.generate(
        {"max_tokens": max_tokens, "temperature": 0.0, "ignore_eos": True}
    )
    snapshot = fenced.lifecycle.snapshot(fenced.key)
    if snapshot is None or snapshot.state != DropOperationState.PLAN_CONSUMED:
        raise RuntimeError(f"unexpected final snapshot: {snapshot}")
    metrics = fenced.lifecycle.metrics
    if metrics.effects_applied != 1 or metrics.remote_completions != 1:
        raise RuntimeError(f"unexpected lifecycle metrics: {metrics}")

    evidence = {
        "served_model_name": served_model_name,
        "hf_model_name": hf_model_name,
        "cache_model_name": cache_model_name,
        "request_stream_id": request_stream.request_stream_id,
        "source_cached_tokens": fenced.source_tokens,
        "kept_cached_tokens": fenced.kept_tokens,
        "drop_ratio": 1.0 - (fenced.kept_tokens / fenced.source_tokens),
        "prefill_output_tokens": prefill.output_tokens,
        "decode_output_tokens": decode.output_tokens,
        "decode_text_preview": request_stream.output_text[:160],
        "operation_generation": fenced.key.request_generation,
        "live_generation_after_reuse": fenced.lifecycle.request_generation,
        "final_state": snapshot.state.value,
        "source_reusable": snapshot.source_reusable,
        "effect_applied": snapshot.effect_applied,
        "duplicate_completion": duplicate.value,
        "late_old_generation_completion": late.value,
        "metrics": asdict(metrics),
    }
    print(json.dumps(evidence, indent=2))
finally:
    ctx.close()